# Estrazione di Feature Esoplanetarie e Predizione

## Introduzione

Questo notebook esplora i dati del **NASA Exoplanet Archive**, il database ufficiale della NASA che raccoglie e cataloga tutti gli esopianeti confermati. L'obiettivo è:

1. Caricare e pulire i dati del NASA Exoplanet Archive
2. Selezionare le feature più rilevanti per l'analisi
3. Applicare un modello di Deep Learning (CNN 1D) per classificazione

## Nota importante sul target

Il dataset contiene **solo esopianeti** (tutti confermati). Non avendo esempi di "non-pianeti", la classificazione binaria non è significativa in questo contesto. Il target viene impostato a 0 per tutti i campioni, quindi il modello impara semplicemente a predire sempre 0 — ottenendo accuratezza 100% ma **senza alcun potere predittivo**.

In un contesto reale servirebbero:
- Dati di stelle **senza** pianeti rilevati (classe negativa)
- Oppure un approccio **non supervisionato** (clustering, anomaly detection)

## Pipeline
- Caricamento e pulizia dati
- Selezione delle feature numeriche
- Suddivisione train/test
- Modello CNN 1D (a scopo dimostrativo)
- Valutazione e critica dei risultati

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam

plt.rcParams['figure.figsize'] = (12, 5)

## 1. Caricamento del Dataset NASA Exoplanet Archive

Il file CSV contiene metadati su esopianeti confermati. Le prime righe sono commenti (con prefisso `#`) che documentano le colonne.

In [ ]:
file_path = "./Data/Nasa_Exoplanet_Archive.csv"

try:
    df = pd.read_csv(file_path, comment='#', sep=',', on_bad_lines='skip', encoding='utf-8')
    print(f"Dataset caricato: {df.shape[0]} righe, {df.shape[1]} colonne")
except FileNotFoundError:
    print(f"Errore: File '{file_path}' non trovato.")
except Exception as e:
    print(f"Errore nel caricamento: {e}")

### Colonne del dataset

| Colonna | Descrizione |
|---------|-------------|
| `pl_name` | Nome del pianeta |
| `pl_orbper` | Periodo orbitale [giorni] |
| `pl_orbsmax` | Semi-asse maggiore [au] |
| `pl_bmasse` | Massa [Masse terrestri] |
| `pl_bmassj` | Massa [Masse gioviane] |
| `pl_orbeccen` | Eccentricità orbitale |
| `pl_insol` | Flusso di insolazione [Flusso terrestre] |
| `pl_eqt` | Temperatura di equilibrio [K] |
| `ra` / `dec` | Coordinate equatoriali [gradi] |
| `sy_dist` | Distanza [pc] |

In [ ]:
df.head()

## 2. Selezione delle Feature

Selezioniamo le colonne numeriche più rilevanti per l'analisi, escludendo quelle testuali o ridondanti.

In [ ]:
selected_columns = [
    'pl_orbper', 'pl_orbsmax', 'pl_bmasse',
    'pl_bmassj', 'pl_orbeccen', 'pl_insol', 'pl_eqt',
    'ra', 'dec', 'sy_dist'
]

df_selected = df[selected_columns].copy()
print(f"Feature selezionate: {df_selected.shape[1]}")

## 3. Pulizia dei dati

Rimuoviamo le righe con valori NaN. La pulizia è fondamentale in quanto molti oggetti hanno misurazioni incomplete.

In [ ]:
df_cleaned = df_selected.dropna()
print(f"Righe dopo dropna: {df_cleaned.shape[0]} (erano {df_selected.shape[0]})")
df_cleaned.info()

## 4. Preparazione del target

**Problema noto**: Il dataset contiene solo esopianeti, quindi non abbiamo una classe negativa. Per scopi dimostrativi, assegniamo target = 0 a tutti i campioni. In un contesto reale servirebbero dati di stelle **senza** pianeti rilevati (classe 1) per addestrare un classificatore binario.

In [ ]:
df_cleaned['target'] = 0

X = df_cleaned.drop('target', axis=1).to_numpy(dtype=np.float64)
y = df_cleaned['target'].to_numpy(dtype=np.int64)

print(f"Feature matrix shape: {X.shape}")
print(f"Target distribution: {np.bincount(y)} (tutti {y[0]})")

## 5. Suddivisione Train/Test

Dividiamo il dataset in 80% training e 20% test.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

print(f"Train: {X_train.shape[0]} samples, shape {X_train.shape}")
print(f"Test: {X_test.shape[0]} samples, shape {X_test.shape}")

## 6. Modello CNN 1D

Definiamo una CNN 1D per classificazione binaria. **Nota**: la CNN non è la scelta ideale per dati tabulari come questi. Una rete completamente connessa (Dense) o modelli ensemble (Random Forest, XGBoost) sarebbero più appropriati. Qui la usiamo a scopo dimostrativo didattico.

In [ ]:
def create_cnn_model(input_shape):
    model = Sequential([
        Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape),
        MaxPooling1D(pool_size=2),
        Dropout(0.25),
        Conv1D(filters=128, kernel_size=3, activation='relu'),
        MaxPooling1D(pool_size=2),
        Dropout(0.25),
        Flatten(),
        Dense(128, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

model = create_cnn_model((X_train.shape[1], 1))
model.summary()

## 7. Addestramento

Il modello viene addestrato per 10 epoche. Poiché il target è tutto zero, la rete impara rapidamente a predire sempre 0, raggiungendo il 100% di accuratezza — ma senza alcun potere predittivo reale.

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=32,
    verbose=1
)

## 8. Valutazione

Le metriche mostrano 0.0 per precision e recall perché non ci sono predizioni positive — conseguenza diretta del target uniforme.

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int)

print("Report di classificazione:")
print(classification_report(y_test, y_pred, zero_division=0))
print(f"Predizioni uniche nel test set: {np.unique(y_pred)}")

## Conclusioni e Miglioramenti Possibili

### Problemi identificati
1. **Target non bilanciato**: Il dataset contiene solo esopianeti (tutti positivi). La classificazione binaria richiede esempi di entrambe le classi.
2. **CNN su dati tabulari**: Le CNN sono progettate per dati con struttura spaziale/temporale locale. Per dati tabulari, reti fully-connected o modelli tabulari (XGBoost, Random Forest) sono più indicati.
3. **Standardizzazione assente**: Le feature non sono normalizzate. `StandardScaler` di sklearn migliorerebbe la convergenza.

### Possibili approcci alternativi
- **Anomaly detection**: Usare tecniche non supervisionate (Isolation Forest, One-Class SVM) per identificare pianeti anomali.
- **Clustering**: Raggruppare gli esopianeti in categorie fisiche (Giove caldi, super-Terre, Nettuniani).
- **Regressione**: Predire proprietà continue (massa, periodo) invece di classificare.
- **Bilanciamento**: Aggiungere dati di stelle senza pianeti rilevati da survey come Kepler o TESS.